<a href="https://colab.research.google.com/github/mohamadatashfaraz4-netizen/master-thesis/blob/main/4_Klassische_Verfahren_und_Werkzeuge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

!pip install -q openpyxl

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Excel-Datei laden
df_original = pd.read_excel("credit_card_corrupted_all_errors.xlsx")

# Ausgangszustand dokumentieren
rows_before = len(df_original)
columns_before = df_original.shape[1]
missing_before = int(df_original.isna().sum().sum())
duplicates_before = int(df_original.duplicated().sum())

print("Rows before cleaning:", rows_before)
print("Columns before cleaning:", columns_before)
print("Missing values before cleaning:", missing_before)
print("Duplicates before cleaning:", duplicates_before)

# Kopie für klassische Bereinigung erstellen
df = df_original.copy()

# Spaltennamen standardisieren
df.columns = (
    df.columns
      .astype(str)
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

# Zielspalte definieren
target_col = "default_payment_next_month"

# Prüfen, ob die Zielspalte vorhanden ist
if target_col not in df.columns:
    raise ValueError(f"Target column not found: {target_col}")

# Zeilen mit fehlender Zielvariable entfernen
rows_before_target_cleaning = len(df)
df = df.dropna(subset=[target_col])
rows_removed_missing_target = rows_before_target_cleaning - len(df)

# Doppelte Zeilen entfernen
duplicates_before_cleaning = int(df.duplicated().sum())
df = df.drop_duplicates()

# Numerische und kategoriale Spalten trennen
numeric_cols = df.select_dtypes(include="number").columns
categorical_cols = df.select_dtypes(exclude="number").columns

# Fehlende numerische Werte durch den Median ersetzen
num_imputer = SimpleImputer(strategy="median")
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

# Fehlende kategoriale Werte durch "unknown" ersetzen
df[categorical_cols] = df[categorical_cols].fillna("unknown")

# Zielvariable wieder als Integer setzen
df[target_col] = df[target_col].astype(int)

# Zustand nach der Bereinigung dokumentieren
rows_after = len(df)
columns_after = df.shape[1]
missing_after = int(df.isna().sum().sum())
duplicates_after = int(df.duplicated().sum())

print("Rows after cleaning:", rows_after)
print("Columns after cleaning:", columns_after)
print("Missing values after cleaning:", missing_after)
print("Duplicates after cleaning:", duplicates_after)

# Bereinigungsbericht erstellen
cleaning_report = pd.DataFrame({
    "metric": [
        "rows_before",
        "rows_after",
        "columns_before",
        "columns_after",
        "missing_values_before",
        "missing_values_after",
        "duplicates_before",
        "duplicates_after",
        "duplicates_removed",
        "rows_removed_missing_target",
    ],
    "value": [
        rows_before,
        rows_after,
        columns_before,
        columns_after,
        missing_before,
        missing_after,
        duplicates_before,
        duplicates_after,
        duplicates_before_cleaning,
        rows_removed_missing_target,
    ],
})

print("\nCleaning report:")
print(cleaning_report)

# Merkmale und Zielvariable trennen
X = df.drop(columns=[target_col])
y = df[target_col]

# Daten in Trainings- und Testdatensatz aufteilen
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42,
    stratify=y
)

# Klassische Baseline-Pipeline erstellen
baseline = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(n_estimators=300, random_state=42)
)

# Baseline-Modell trainieren
baseline.fit(X_train, y_train)

# Vorhersagen und Wahrscheinlichkeiten erzeugen
predictions = baseline.predict(X_test)
probabilities = baseline.predict_proba(X_test)[:, 1]

# Modellleistung bewerten
accuracy = accuracy_score(y_test, predictions)
roc_auc = roc_auc_score(y_test, probabilities)

print("\nModel performance:")
print("Accuracy:", accuracy)
print("ROC AUC:", roc_auc)

# Modellbericht erstellen
model_report = pd.DataFrame({
    "metric": ["accuracy", "roc_auc"],
    "value": [accuracy, roc_auc]
})

# Dateien speichern
cleaned_file = "chapter04_cleaned_credit_card_data.xlsx"
cleaning_report_file = "chapter04_cleaning_report.xlsx"
model_report_file = "chapter04_baseline_model_report.xlsx"

df.to_excel(cleaned_file, index=False)
cleaning_report.to_excel(cleaning_report_file, index=False)
model_report.to_excel(model_report_file, index=False)

# Dateien herunterladen
files.download(cleaned_file)
files.download(cleaning_report_file)
files.download(model_report_file)

print("\nSaved files:")
print(cleaned_file)
print(cleaning_report_file)
print(model_report_file)

Saving credit_card_corrupted_all_errors.xlsx to credit_card_corrupted_all_errors.xlsx
Rows before cleaning: 30300
Columns before cleaning: 25
Missing values before cleaning: 36210
Duplicates before cleaning: 282
Rows after cleaning: 28492
Columns after cleaning: 25
Missing values after cleaning: 0
Duplicates after cleaning: 0

Cleaning report:
                        metric  value
0                  rows_before  30300
1                   rows_after  28492
2               columns_before     25
3                columns_after     25
4        missing_values_before  36210
5         missing_values_after      0
6            duplicates_before    282
7             duplicates_after      0
8           duplicates_removed    267
9  rows_removed_missing_target   1541

Model performance:
Accuracy: 0.7960225459959588
ROC AUC: 0.7366983179405843


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Saved files:
chapter04_cleaned_credit_card_data.xlsx
chapter04_cleaning_report.xlsx
chapter04_baseline_model_report.xlsx
